# Решения: Практика численного градиентного спуска в 1D

**Для преподавателя.** Ниже по разделам разобраны все задачи `lesson.ipynb` и `homework.ipynb`. Не выдавать до сдачи.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def loss(x):
    return (x - 2.5) ** 2 + 0.2

def derivative_central(fun, x, h=1e-4):
    return float((fun(x + h) - fun(x - h)) / (2 * h))


## Урок. 1. Проверка направления шага

In [ ]:
x0, eta = -1.0, 0.2
gradient0 = derivative_central(loss, x0)
x1 = x0 - eta * gradient0
assert loss(x1) < loss(x0)
print(x0, gradient0, x1, loss(x1))


## Урок. 2. Один шаг как функция

In [ ]:
def gd_step(fun, x, eta):
    return float(x - eta * derivative_central(fun, x))

left_step = gd_step(loss, -1.0, 0.1)
right_step = gd_step(loss, 4.0, 0.1)
assert left_step > -1.0 and right_step < 4.0
print(left_step, right_step)


## Урок. 3. Полная траектория

In [ ]:
def gradient_descent_1d(fun, start, eta=0.1, steps=25):
    x = float(start)
    path_x = [x]
    path_loss = [float(fun(x))]
    for _ in range(steps):
        x = gd_step(fun, x, eta)
        path_x.append(x)
        path_loss.append(float(fun(x)))
    return path_x, path_loss

path_x, path_loss = gradient_descent_1d(loss, -1.0, eta=0.2, steps=20)
assert len(path_x) == len(path_loss) == 21 and path_loss[-1] < path_loss[0]
print(path_x[-1], path_loss[-1])


## Урок. 4. Инварианты траектории

In [ ]:
is_finite = bool(np.all(np.isfinite(path_loss)))
non_increasing = bool(np.all(np.diff(path_loss) <= 1e-10))
assert is_finite and non_increasing
print(is_finite, non_increasing)


## Урок. 5. Сравнение трех шагов обучения

In [ ]:
eta_values = [0.03, 0.2, 0.9]
final_losses = [
    gradient_descent_1d(loss, -1.0, eta=value, steps=25)[1][-1]
    for value in eta_values
]
assert len(final_losses) == len(eta_values)
print(list(zip(eta_values, final_losses)))


## Урок. 6. График траекторий

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for value in eta_values:
    _, losses = gradient_descent_1d(loss, -1.0, eta=value, steps=25)
    ax.plot(losses, marker=".", label=f"eta={value}")
ax.set(xlabel="step", ylabel="loss", title="Как eta меняет траекторию")
ax.legend()
assert len(ax.lines) == len(eta_values)
plt.show()


## Урок. 7. Критерий ранней остановки

In [ ]:
def gradient_descent_until(fun, start, eta=0.1, max_steps=100, tolerance=1e-5):
    x = float(start)
    path_x, path_loss = [x], [float(fun(x))]
    for _ in range(max_steps):
        gradient = derivative_central(fun, x)
        if abs(gradient) < tolerance:
            break
        x = float(x - eta * gradient)
        path_x.append(x)
        path_loss.append(float(fun(x)))
    return path_x, path_loss

short_x, short_loss = gradient_descent_until(loss, -1.0, eta=0.2)
assert abs(derivative_central(loss, short_x[-1])) < 1e-5
print(len(short_x), short_x[-1], short_loss[-1])


## Урок. 8. Самостоятельно: диагностический отчёт

In [ ]:
def diagnose_run(losses):
    values = np.asarray(losses, dtype=float)
    if not np.all(np.isfinite(values)):
        return "invalid"
    if np.any(np.diff(values) > 1e-9):
        return "unstable"
    if len(values) >= 2 and abs(values[-1] - values[-2]) < 1e-6:
        return "converged"
    return "slow"

diagnoses = [diagnose_run(gradient_descent_1d(loss, -1, e, 25)[1]) for e in eta_values]
ETA_NOTE = (
    "Шаг eta выбирают по траектории, а не по одному финальному числу. Малый шаг может быть "
    "стабильным, но медленным; слишком большой дает рост или колебания loss. Для этой квадратичной "
    "функции eta=0.2 быстро уменьшает loss без нарушения монотонности."
)
assert len(ETA_NOTE) >= 160
print(diagnoses)
print(ETA_NOTE)


## ДЗ. Данные и функции

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def loss(x):
    return (x - 2.5) ** 2 + 0.2

def derivative_central(fun, x, h=1e-4):
    return float((fun(x + h) - fun(x - h)) / (2 * h))

def rugged_loss(x):
    return 0.15 * (x - 4.0) ** 2 + 0.3 * np.sin(2.2 * x) + 1.0


## ДЗ. Закрепление: перенесите GD

In [ ]:
def gd(fun, start, eta=0.08, steps=40):
    x = float(start)
    path_x, losses = [x], [float(fun(x))]
    for _ in range(steps):
        x -= eta * derivative_central(fun, x)
        path_x.append(float(x))
        losses.append(float(fun(x)))
    return path_x, losses

path_a, loss_a = gd(rugged_loss, -2.0)
assert len(path_a) == len(loss_a) == 41
print(path_a[-1], loss_a[-1])


## ДЗ. База: разные старты

In [ ]:
starts = [-2.0, 1.0, 7.0]
runs = [gd(rugged_loss, start) for start in starts]
end_x = [path[-1] for path, _ in runs]
end_loss = [losses[-1] for _, losses in runs]
assert len(end_x) == len(starts)
print(list(zip(starts, end_x, end_loss)))


## ДЗ. Углубление: сетка eta

In [ ]:
eta_grid = [0.01, 0.03, 0.08, 0.15, 0.3]
eta_scores = [(eta, gd(rugged_loss, -2.0, eta=eta)[1][-1]) for eta in eta_grid]
best_eta = min(eta_scores, key=lambda item: item[1])[0]
assert best_eta in eta_grid
print(eta_scores, best_eta)


## ДЗ. Вызов: локальные минимумы

In [ ]:
LOCAL_NOTE = (
    "На неровной функции градиентный спуск использует только локальный наклон и не видит весь "
    "рельеф. Поэтому два старта могут попасть в разные впадины и закончить с разными loss. "
    "Меньший финальный loss среди нескольких запусков лучше в этом эксперименте, но перебор "
    "трех стартов не доказывает нахождение глобального минимума. Это ограничение метода, а не "
    "ошибка реализации."
)
COUNTEREXAMPLE_READY = True
assert len(LOCAL_NOTE) >= 240 and COUNTEREXAMPLE_READY
print(LOCAL_NOTE)
